## Aqui foi feito tanto a conversão quanto as análises exploratórias

### Bibliotecas usadas para fazer as conversões

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, r2_score

### 1 - Filtragem do valores desejados

In [ ]:
df_matriculas = pd.read_csv('alunos_matriculados/matriculas_por_estado2010.csv')  
df_taxas = pd.read_csv('rendimento_escolar/Dados Tratados/taxas_abandono_por_estado2010.csv')   


df_taxas = df_taxas.rename(columns={'Estado': 'NO_UF'})


df_final = pd.merge(df_matriculas, df_taxas, on='NO_UF', how='left')


df_final['Abandono_EF'] = (df_final['Total_Fundamental'] * df_final['Taxa_Abandono_EF2010']) / 100
df_final['Abandono_EM'] = (df_final['Total_Medio'] * df_final['Taxa_Abandono_EM2010']) / 100


df_final['Abandono_EF'] = df_final['Abandono_EF'].round().fillna(0).astype(int)
df_final['Abandono_EM'] = df_final['Abandono_EM'].round().fillna(0).astype(int)

df_final['Total_Abandonos'] = df_final['Abandono_EF'] + df_final['Abandono_EM']

df_final = df_final.rename(columns={'NO_UF': 'Estado'})

df_final.to_csv('abandono_escolar_por_estado2010.csv', index=False, encoding='utf-8-sig')


Arquivo 'abandono_escolar_por_estado.csv' gerado com sucesso!

Resumo estatístico:
        Abandono_EF    Abandono_EM  Total_Abandonos
count     27.000000      27.000000        27.000000
mean   19874.851852   38765.333333     58640.185185
std    21529.108240   32203.625603     50129.249166
min     1099.000000    1514.000000      2613.000000
25%     4485.500000   12649.500000     17669.000000
50%    13949.000000   32148.000000     50595.000000
75%    27131.500000   55263.000000     83467.000000
max    80850.000000  114268.000000    195118.000000

5 primeiros registros:
     Estado  Total_Fundamental  Total_Medio  Total_Geral  \
0      Acre           166200.0      36295.0     202495.0   
1   Alagoas           630889.0     130247.0     761136.0   
2     Amapá           144597.0      37871.0     182468.0   
3  Amazonas           771963.0     162113.0     934076.0   
4     Bahia          2450008.0     589012.0    3039020.0   

   Taxa_Abandono_EF2010  Taxa_Abandono_EM2010  Abandono_EF  Aba

tratamento dos alores das variaveis para que tenham apenas 2 casas decimais

### 2 - Convertendo dados do bolsa família em csv

#### baixando os dados do bolsa familias e convertendo para o formato CSV

In [ ]:


url = "https://aplicacoes.mds.gov.br/sagi/servicos/misocial/?fq=anomes_s:2023*&fl=codigo_ibge%2Canomes_s%2Cqtd_familias_beneficiarias_bolsa_familia_s%2Cvalor_repassado_bolsa_familia_s%2Cpbf_vlr_medio_benef_f&fq=valor_repassado_bolsa_familia_s%3A*&q=*%3A*&rows=100000&sort=anomes_s%20desc%2C%20codigo_ibge%20asc&wt=csv"

dados_bf = pd.read_csv(url, encoding='latin1', sep=';')

dados_bf.to_csv('valorRepassado_familia_2023.csv', index=False, encoding='utf-8-sig')

print("Arquivo salvo com sucesso!")

dados = pd.read_csv('valorRepassado_familia_2023.csv', sep=';', encoding='latin1')

dados = dados.replace({'"': ''}, regex=True)

dados.to_csv('valorRepassado_familia_2023.csv', index=False, sep=';', encoding='utf-8-sig')

print("Arquivo corrigido salvo sem aspas!")


Arquivo salvo com sucesso!
Arquivo corrigido salvo sem aspas!


In [ ]:
uf_map = {
    11: {'sigla': 'RO', 'nome': 'Rondônia'},
    12: {'sigla': 'AC', 'nome': 'Acre'},
    13: {'sigla': 'AM', 'nome': 'Amazonas'},
    14: {'sigla': 'RR', 'nome': 'Roraima'},
    15: {'sigla': 'PA', 'nome': 'Pará'},
    16: {'sigla': 'AP', 'nome': 'Amapá'},
    17: {'sigla': 'TO', 'nome': 'Tocantins'},
    21: {'sigla': 'MA', 'nome': 'Maranhão'},
    22: {'sigla': 'PI', 'nome': 'Piauí'},
    23: {'sigla': 'CE', 'nome': 'Ceará'},
    24: {'sigla': 'RN', 'nome': 'Rio Grande do Norte'},
    25: {'sigla': 'PB', 'nome': 'Paraíba'},
    26: {'sigla': 'PE', 'nome': 'Pernambuco'},
    27: {'sigla': 'AL', 'nome': 'Alagoas'},
    28: {'sigla': 'SE', 'nome': 'Sergipe'},
    29: {'sigla': 'BA', 'nome': 'Bahia'},
    31: {'sigla': 'MG', 'nome': 'Minas Gerais'},
    32: {'sigla': 'ES', 'nome': 'Espírito Santo'},
    33: {'sigla': 'RJ', 'nome': 'Rio de Janeiro'},
    35: {'sigla': 'SP', 'nome': 'São Paulo'},
    41: {'sigla': 'PR', 'nome': 'Paraná'},
    42: {'sigla': 'SC', 'nome': 'Santa Catarina'},
    43: {'sigla': 'RS', 'nome': 'Rio Grande do Sul'},
    50: {'sigla': 'MS', 'nome': 'Mato Grosso do Sul'},
    51: {'sigla': 'MT', 'nome': 'Mato Grosso'},
    52: {'sigla': 'GO', 'nome': 'Goiás'},
    53: {'sigla': 'DF', 'nome': 'Distrito Federal'}
}

ano = 2021 

df = pd.read_csv("valorRepassado_familia_2021.csv")  

df['cod_uf'] = df['ibge'].astype(str).str[:2].astype(int)

df['uf_sigla'] = df['cod_uf'].map(lambda x: uf_map[x]['sigla'])
df['uf_nome'] = df['cod_uf'].map(lambda x: uf_map[x]['nome'])

df_estado = df.groupby(['uf_sigla', 'uf_nome']).agg({
    'valor_repassado_bolsa_familia': 'sum',
    'qtd_familias_beneficiarias_bolsa_familia': 'sum'
}).reset_index()

df_estado['valor_medio_bf'] = df_estado['valor_repassado_bolsa_familia'] / df_estado['qtd_familias_beneficiarias_bolsa_familia']
df_estado['ano'] = ano  

df_estado = df_estado.sort_values('uf_sigla')

resultado = df_estado[['ano', 'uf_sigla', 'uf_nome', 'qtd_familias_beneficiarias_bolsa_familia', 'valor_repassado_bolsa_familia', 'valor_medio_bf']]

nome_arquivo_saida = f'bolsa_familia_por_estado_{ano}.csv'
resultado.to_csv(nome_arquivo_saida, index=False, float_format='%.2f')

print(f"Arquivo salvo: {nome_arquivo_saida}")
print(resultado.head())

Arquivo salvo: bolsa_familia_por_estado_2021.csv
    ano uf_sigla   uf_nome  qtd_familias_beneficiarias_bolsa_familia  \
0  2021       AC      Acre                                  907936.0   
1  2021       AL   Alagoas                                 4132720.0   
2  2021       AM  Amazonas                                 4059479.0   
3  2021       AP     Amapá                                  758406.0   
4  2021       BA     Bahia                                18580643.0   

   valor_repassado_bolsa_familia  valor_medio_bf  
0                   1.844038e+08      203.102209  
1                   4.780583e+08      115.676420  
2                   6.279694e+08      154.692105  
3                   1.102775e+08      145.407009  
4                   2.060550e+09      110.897653  


### 3 - Adicionar a coluna contendo a sigla de cada estado para facilitar a manipulação posterior dos dados, converte os dados para que os valores de quantidade de familias e o valor medio da bolsa familia sejam agrupados por estados

##### Aqui foi utilizado o bolsa_familia_por_estado de cada ano no mesmo código com reajustes IPCA IBGE de cada ano respectivo em comparação ao poder de compra de 2024 (valor mais atual), trazendo mais proximidade na análise.

In [ ]:
df = pd.read_csv('bolsa_familia/dadosLimpos/bolsa_familia_por_estado_2021.csv')  


if 'valor_medio_bf' in df.columns:

    df['valor_medio_bf_ajustado'] = (df['valor_medio_bf'] * 1.02985750).round(2)
    

    df.to_csv('bolsa_familia_por_estado_2021.csv', index=False, float_format='%.2f')
    
    print("Ajuste realizado com sucesso! Resultado com 2 casas decimais:")
    print(df[['uf_sigla', 'valor_medio_bf', 'valor_medio_bf_ajustado']].head())
else:
    print("Erro: Coluna 'valor_medio_bf' não encontrada.")
    print("Colunas disponíveis:", list(df.columns))

Ajuste realizado com sucesso! Resultado com 2 casas decimais:
  uf_sigla  valor_medio_bf  valor_medio_bf_ajustado
0       AC          203.10                   209.16
1       AL          115.68                   119.13
2       AM          154.69                   159.31
3       AP          145.41                   149.75
4       BA          110.90                   114.21


#### Filtra os dados importantes, esse processo é repetido por cada ano 

In [ ]:
acress_df = merged_df[['UF', 'Resultado_Multiplicacao_EF2008', 'Resultado_Multiplicacao_EM2008']]
agrupado_por_uf = acress_df.groupby('UF', as_index=False).sum()
agrupado_por_uf

### Mesclar todos os dados do abandono escolar em apenas um dataframe 

In [ ]:
ano2023 = pd.read_csv("total_abandono_alunos_estados/abandono_escolar_por_estado2023.csv")

### recorta as colunas que são importantes

In [ ]:
novo2023= ano2023[['Estado','Abandono_EF','Abandono_EM']]

### mescla os dados em um dataframe unico

In [ ]:
merged_df = pd.merge(dados, novo2023, left_on='Estado', right_on='Estado', how='outer')

### salva esse dataframe que contem os dados de todos os anos como um novo arquivo

In [ ]:
totalTotal.to_csv("totais por ano.csv", sep=",", index=False)

### filtra as colunas que são importantes e as renomeia

In [ ]:
novobolsa2023= bolsa2023[['uf_nome','valor_medio_estadual','qtd_familias_beneficiarias_bolsa_familia_ajustado']]
novobolsa2023.rename(columns={'uf_nome':'Estado'}, inplace=True)

### salva em um dataframe os dados de todos os anos

In [ ]:
merged_df1 = pd.merge(dados1, novobolsa2023, left_on='Estado', right_on='Estado', how='outer')

In [ ]:
totalTotal= merged_df1

### salva esse dataframe como sendo um arquivo

In [ ]:
totalTotal.to_csv("totais por ano.csv", sep=",", index=False)

### 4 - foram incluídas variáveis de taxa de Desocupação, nivel de Ocupação e taxa de participação. Para essas variáveis foi realizado um processo de tratamento para filtrar os dados importantes

In [ ]:
taxas = pd.read_csv("INDICADORES_TABELA2.csv")

df_segundo_trimestre = taxas[taxas['Trimestre'] == 2]
colunas_desejadas = ['UF', 'Ano', 'Trimestre', 'Taxa_Desocupacao', 'Nivel_Ocupacao', 'Taxa_Participacao']
df_filtrado = df_segundo_trimestre[colunas_desejadas]

df_filtrado = df_filtrado.reset_index(drop=True)

mapa_estados = {
    "Acre": "AC", "Alagoas": "AL", "Amapá": "AP", "Amazonas": "AM", "Bahia": "BA",
    "Ceará": "CE", "Espírito Santo": "ES", "Goiás": "GO", "Maranhão": "MA", "Mato Grosso": "MT",
    "Mato Grosso do Sul": "MS", "Minas Gerais": "MG", "Pará": "PA", "Paraíba": "PB", "Paraná": "PR",
    "Pernambuco": "PE", "Piauí": "PI", "Rio de Janeiro": "RJ", "Rio Grande do Norte": "RN", "Rio Grande do Sul": "RS",
    "Rondônia": "RO", "Roraima": "RR", "Santa Catarina": "SC", "São Paulo": "SP", "Sergipe": "SE",
    "Tocantins": "TO", "Distrito Federal": "DF"
}

df_filtrado['UF'] = df_filtrado['UF'].apply(lambda x: mapa_estados.get(x, x)) 

df_filtrado.to_csv("TaxasDeDesocupacaoOcupacaoParticipacao.csv",index=False)